In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

In [ ]:
df = pd.read_csv("../datasets/united.csv", index_col = 0)

df["processing_time"] = pd.to_datetime(df["processing_time"])
df = df.sort_values(["well_id", "processing_time"]).reset_index(drop=True)

In [5]:
df

,processing_time,rotation,pressure_axis,pressure_rotation,well_id,speed
0,2025-10-07 22:18:14.495,104.892,12781,13144,25512,0.02020
1,2025-10-07 22:18:19.239,101.430,18971,18936,25512,0.03636
2,2025-10-07 22:18:24.208,101.430,20153,18184,25512,0.03030
3,2025-10-07 22:18:29.385,102.714,20162,17445,25512,0.03030
4,2025-10-07 22:19:22.224,103.110,18591,19741,25512,0.02424
...,...,...,...,...,...,...
415044,2025-10-07 22:14:55.384,104.598,10071,13434,25512,0.03636
415045,2025-10-07 22:14:59.667,105.978,12308,12992,25512,0.03636
415046,2025-10-07 22:15:05.094,105.978,12363,15220,25512,0.01515
415047,2025-10-07 22:15:10.387,102.420,13499,14416,25512,0.01212


In [ ]:
features = ["rotation", "pressure_axis", "pressure_rotation", "speed"]

X = df[features].replace([np.inf, -np.inf], np.nan).dropna()
df_clustered = df.loc[X.index].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X.shape

In [ ]:
random_state = 42
fit_sample_size = min(50_000, len(X_scaled))
metric_sample_size = min(10_000, len(X_scaled))

rng = np.random.default_rng(random_state)
fit_idx = rng.choice(len(X_scaled), size=fit_sample_size, replace=False)
metric_idx = rng.choice(len(X_scaled), size=metric_sample_size, replace=False)

X_fit = X_scaled[fit_idx]
X_metric = X_scaled[metric_idx]

component_range = range(2, 9)
results = []
models = {}

for n_components in component_range:
    gmm = GaussianMixture(
        n_components=n_components,
        covariance_type="full",
        n_init=3,
        random_state=random_state,
    )
    gmm.fit(X_fit)

    metric_labels = gmm.predict(X_metric)
    results.append(
        {
            "n_components": n_components,
            "silhouette_sample": silhouette_score(X_metric, metric_labels),
            "calinski_harabasz_sample": calinski_harabasz_score(X_metric, metric_labels),
            "davies_bouldin_sample": davies_bouldin_score(X_metric, metric_labels),
            "bic_fit_sample": gmm.bic(X_fit),
            "aic_fit_sample": gmm.aic(X_fit),
            "avg_log_likelihood_fit_sample": gmm.score(X_fit),
        }
    )
    models[n_components] = gmm

gmm_metrics = pd.DataFrame(results)
gmm_metrics

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
metric_specs = [
    ("silhouette_sample", "Silhouette (higher is better)"),
    ("calinski_harabasz_sample", "Calinski-Harabasz (higher is better)"),
    ("davies_bouldin_sample", "Davies-Bouldin (lower is better)"),
    ("bic_fit_sample", "BIC (lower is better)"),
    ("aic_fit_sample", "AIC (lower is better)"),
    ("avg_log_likelihood_fit_sample", "Avg log likelihood (higher is better)"),
]

for ax, (metric, title) in zip(axes.ravel(), metric_specs):
    ax.plot(gmm_metrics["n_components"], gmm_metrics[metric], marker="o")
    ax.set_title(title)
    ax.set_xlabel("GMM components")
    ax.set_ylabel(metric)
    ax.grid(alpha=0.3)

plt.show()

In [ ]:
best_n_components = int(gmm_metrics.loc[gmm_metrics["bic_fit_sample"].idxmin(), "n_components"])
best_gmm = models[best_n_components]

df_clustered["cluster"] = best_gmm.predict(X_scaled)
df_clustered["cluster_probability"] = best_gmm.predict_proba(X_scaled).max(axis=1)

best_n_components, df_clustered[["cluster", "cluster_probability"]].head()

In [ ]:
plot_sample_size = min(20_000, len(X_scaled))
plot_idx = rng.choice(len(X_scaled), size=plot_sample_size, replace=False)

pca = PCA(n_components=2, random_state=random_state)
X_pca = pca.fit_transform(X_scaled[plot_idx])
plot_labels = df_clustered.iloc[plot_idx]["cluster"].to_numpy()

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=plot_labels,
    cmap="tab10",
    s=8,
    alpha=0.45,
    linewidths=0,
)
ax.set_title(f"GMM clusters in PCA space (n_components={best_n_components})")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.grid(alpha=0.25)
fig.colorbar(scatter, ax=ax, label="cluster")
plt.show()

In [ ]:
cluster_profile = (
    df_clustered.groupby("cluster")[features + ["cluster_probability"]]
    .agg(["count", "mean", "std", "median"])
    .round(3)
)

cluster_profile

In [ ]:
cluster_means = df_clustered.groupby("cluster")[features].mean()
cluster_means_scaled = pd.DataFrame(
    scaler.transform(cluster_means),
    index=cluster_means.index,
    columns=features,
)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(cluster_means_scaled, aspect="auto", cmap="coolwarm")
ax.set_title("Cluster feature profiles (scaled means)")
ax.set_xlabel("feature")
ax.set_ylabel("cluster")
ax.set_xticks(np.arange(len(features)), labels=features, rotation=35, ha="right")
ax.set_yticks(np.arange(len(cluster_means_scaled.index)), labels=cluster_means_scaled.index)
fig.colorbar(im, ax=ax, label="scaled mean")
plt.show()